# Session 3: many floors, many mode shapes
**A. Real-world problem.** A ten-storey frame does not sway as one lump. Each floor is a mass,
each storey of columns is a spring, and the whole stack can bend in ten different patterns
(*modes*), each with its own period. We shake it with a synthetic earthquake record, solve
it two independent ways (modal superposition and direct Newmark-beta integration), check
energy, and then watch a weak ("soft") storey ruin the drift.

**Plan.** (1) build M and K, (2) solve K phi = w^2 M phi and check against the analytic
uniform-building formula, (3) modal participation and effective mass, (4) a seeded synthetic
ground record, (5) two time-domain solvers that must agree, (6) drift and base shear.

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"
N_FLOORS = 10
M_FLOOR = 2.0e5          # kg per floor (200 tonnes)
K_STOREY = 3.5e8         # N/m per storey (all columns of one storey together)
ZETA = 0.05              # damping ratio in every mode

## Step 1: the matrices
Floor j has displacement u_j (relative to the ground). Storey j connects floor j-1 to floor j
(floor 0 is the fixed ground). The equations of motion are  M u'' + C u' + K u = -M r a_g(t),
where r is the vector of ones (the ground pushes every floor the same way) and a_g is the
ground acceleration. K is *tridiagonal*: each floor only feels its two neighbours.

In [ ]:
def shear_building(n=N_FLOORS, m=M_FLOOR, k=K_STOREY):
    """Mass matrix M (diagonal) and stiffness matrix K (tridiagonal). m and k can be scalars or arrays of length n.
    k[j] is the stiffness of storey j+1 (between floor j and floor j+1; floor 0 is the ground)."""
    m = np.broadcast_to(np.asarray(m, float), (n,)).copy()
    k = np.broadcast_to(np.asarray(k, float), (n,)).copy()
    K = np.zeros((n, n))
    for j in range(n):
        K[j, j] += k[j]
        if j + 1 < n:
            K[j, j] += k[j + 1]
            K[j, j + 1] = K[j + 1, j] = -k[j + 1]
    return np.diag(m), K

## Step 2: natural frequencies and mode shapes
Free vibration u = phi cos(w t) gives the generalised eigenproblem  K phi = w^2 M phi.
M is diagonal, so M^(-1/2) exists. Put phi = M^(-1/2) v: then  (M^(-1/2) K M^(-1/2)) v = w^2 v,
an ordinary symmetric eigenproblem that np.linalg.eigh solves. We scale each phi so that
phi^T M phi = 1 (mass-normalised) and flip its sign so the roof moves positive.

In [ ]:
def modes(M, K):
    """Return (w, phi): angular frequencies (rad/s, ascending) and mass-normalised mode shapes (columns)."""
    s = 1.0 / np.sqrt(np.diag(M))
    A = (s[:, None] * K) * s[None, :]
    w2, v = np.linalg.eigh(A)
    phi = s[:, None] * v
    phi = phi * np.where(phi[-1, :] < 0, -1.0, 1.0)
    return np.sqrt(w2), phi


def uniform_analytic(n, m, k):
    """Standard result for equal masses m and equal storey stiffness k, fixed base, free top:
    w_i = 2 sqrt(k/m) sin((2i-1) pi / (2 (2n+1))),  shape_j = sin((2i-1) j pi / (2n+1)), j = 1..n.
    Returns (w, shapes) with the shapes scaled to a roof value of 1 (columns)."""
    i = np.arange(1, n + 1)
    w = 2.0 * np.sqrt(k / m) * np.sin((2 * i - 1) * np.pi / (2 * (2 * n + 1)))
    j = np.arange(1, n + 1)
    sh = np.sin(np.outer(j, (2 * i - 1)) * np.pi / (2 * n + 1))
    return w, sh / sh[-1, :]


def periods(w):
    return 2 * np.pi / w

## Step 3: how much of the building each mode moves
Participation factor  Gamma_i = phi_i^T M r  (mass-normalised shapes). Effective modal mass
M*_i = Gamma_i^2. Together the effective masses add up to the total mass, and the first
mode carries most of it.

In [ ]:
def participation(M, phi):
    r = np.ones(M.shape[0])
    gamma = phi.T @ M @ r
    return gamma, gamma ** 2

## Step 4: a synthetic ground-motion record
**This is not a real earthquake.** It is a sum of 120 sine waves with random (seeded) phases,
amplitudes peaked around a dominant period, and a rise-hold-fade envelope. Real records
differ; the point is to have a repeatable, rich shaking that is centred near 1 s.

In [ ]:
def synthetic_ground(seed=7, t_dom=1.0, dur=24.0, dt=0.01, pga=2.0):
    """Return (t, a_g): ground acceleration in m/s^2, scaled so max |a_g| = pga."""
    rng = np.random.default_rng(seed)
    t = np.arange(0.0, dur + 0.5 * dt, dt)
    f = np.linspace(0.15, 12.0, 120)
    amp = np.exp(-0.5 * (np.log(f * t_dom) / 0.45) ** 2)
    ph = rng.uniform(0, 2 * np.pi, f.size)
    a = (amp[None, :] * np.sin(2 * np.pi * f[None, :] * t[:, None] + ph[None, :])).sum(axis=1)
    env = np.clip(t / 3.0, 0, 1) * np.clip((dur - t) / 6.0, 0, 1)
    a = a * env
    return t, a * (pga / np.abs(a).max())

## Step 5: two independent solvers
**Modal superposition.** Each mode is a one-degree-of-freedom oscillator
q'' + 2 zeta w q' + w^2 q = -Gamma a_g(t), solved *exactly* for a ground record that is
piecewise linear between samples (the Nigam-Jennings recurrence, done here by the exact
2x2 state transition). Then u = sum_i phi_i q_i.

**Direct Newmark-beta.** Step the full matrix equation with the *average acceleration*
method, beta = 1/4 and gamma = 1/2: unconditionally stable, and no numerical damping.
The damping matrix C = M Phi diag(2 zeta w) Phi^T M gives exactly zeta in every mode.

In [ ]:
def damping_matrix(M, phi, w, zeta=ZETA):
    return M @ phi @ np.diag(2 * zeta * w) @ phi.T @ M


def sdof_response(w, zeta, a_g, dt):
    """Relative displacement and velocity of q'' + 2 zeta w q' + w^2 q = -a_g(t); exact for piecewise-linear a_g."""
    wd = w * np.sqrt(1 - zeta ** 2)
    # state z = (q, q'). Homogeneous part: z' = A z. Forcing f = (0, -a_g) is linear in each step.
    A = np.array([[0.0, 1.0], [-w * w, -2 * zeta * w]])
    # matrix exponential of A*dt via the closed form for the underdamped 2x2 case
    e, c, s = np.exp(-zeta * w * dt), np.cos(wd * dt), np.sin(wd * dt)
    Phi = e * (c * np.eye(2) + (s / wd) * (A + zeta * w * np.eye(2)))
    Ainv = np.linalg.inv(A)
    I2 = np.eye(2)
    # z' = A z + b0 + b1 tau, b0 = (0, -a_i), b1 = (0, -(a_{i+1}-a_i)/dt)
    G0 = Ainv @ (Phi - I2)                       # integral of exp(A (dt - s)) ds
    G1 = Ainv @ G0 - dt * Ainv                   # integral of exp(A (dt - s)) s ds (integration by parts)
    n = a_g.size
    q = np.zeros(n)
    v = np.zeros(n)
    z = np.zeros(2)
    for i in range(n - 1):
        b0 = np.array([0.0, -a_g[i]])
        b1 = np.array([0.0, -(a_g[i + 1] - a_g[i]) / dt])
        z = Phi @ z + G0 @ b0 + G1 @ b1
        q[i + 1], v[i + 1] = z
    return q, v


def modal_response(M, K, a_g, dt, zeta=ZETA, nmodes=None):
    """Floor displacements (n_steps, n) by modal superposition with the first `nmodes` modes (default all)."""
    w, phi = modes(M, K)
    gamma, _ = participation(M, phi)
    nm = len(w) if nmodes is None else nmodes
    u = np.zeros((a_g.size, len(w)))
    for i in range(nm):
        q, _ = sdof_response(w[i], zeta, gamma[i] * a_g, dt)
        u += np.outer(q, phi[:, i])
    return u


def newmark(M, C, K, a_g, dt, beta=0.25, gamma=0.5):
    """Direct integration of M u'' + C u' + K u = -M r a_g. Average acceleration: beta = 1/4, gamma = 1/2.
    Returns u, v, a (relative to the ground), each of shape (n_steps, n)."""
    n = M.shape[0]
    r = np.ones(n)
    nt = a_g.size
    u = np.zeros((nt, n))
    v = np.zeros((nt, n))
    a = np.zeros((nt, n))
    a[0] = np.linalg.solve(M, -M @ r * a_g[0] - C @ v[0] - K @ u[0])
    Keff = K + gamma / (beta * dt) * C + M / (beta * dt ** 2)
    Kinv = np.linalg.inv(Keff)
    for i in range(nt - 1):
        p = -M @ r * a_g[i + 1]
        rhs = (p + M @ (u[i] / (beta * dt ** 2) + v[i] / (beta * dt) + (0.5 / beta - 1) * a[i])
               + C @ (gamma / (beta * dt) * u[i] + (gamma / beta - 1) * v[i] + dt * (0.5 * gamma / beta - 1) * a[i]))
        u[i + 1] = Kinv @ rhs
        a[i + 1] = (u[i + 1] - u[i]) / (beta * dt ** 2) - v[i] / (beta * dt) - (0.5 / beta - 1) * a[i]
        v[i + 1] = v[i] + dt * ((1 - gamma) * a[i] + gamma * a[i + 1])
    return u, v, a

## Step 6: what engineers read off
* **Inter-storey drift** = u_j - u_(j-1), the bending of one storey (drift ratio = drift / storey height).
* **Base shear** = the force in the lowest storey = k_1 u_1 (what the foundation must resist).
* **Energy balance**: input energy = kinetic + strain + damping dissipated.

In [ ]:
def storey_drift(u):
    return np.diff(np.hstack([np.zeros((u.shape[0], 1)), u]), axis=1)


def base_shear(u, k1):
    return k1 * u[:, 0]


def energy_balance(M, C, K, u, v, a_g, dt):
    """Return (input, kinetic + strain + damped) at the final step, in joules (J)."""
    r = np.ones(M.shape[0])
    f_in = -(v @ (M @ r)) * a_g                       # power put in by the ground (W)
    e_in = np.concatenate([[0.0], np.cumsum(0.5 * (f_in[1:] + f_in[:-1]) * dt)])
    p_damp = np.einsum("ti,ij,tj->t", v, C, v)
    e_damp = np.concatenate([[0.0], np.cumsum(0.5 * (p_damp[1:] + p_damp[:-1]) * dt)])
    e_kin = 0.5 * np.einsum("ti,ij,tj->t", v, M, v)
    e_strain = 0.5 * np.einsum("ti,ij,tj->t", u, K, u)
    return e_in, e_kin + e_strain + e_damp


def sdof_peak(period, zeta, a_g, dt):
    """Peak |relative displacement| (m) of a one-degree-of-freedom oscillator: one point of the response spectrum."""
    q, _ = sdof_response(2 * np.pi / period, zeta, a_g, dt)
    return float(np.abs(q).max())


STOREY_HEIGHT = 3.5      # m, used to turn drift into a drift ratio


def peaks(k=K_STOREY, zeta=ZETA, seed=7, t_dom=1.0, pga=2.0, n=N_FLOORS, m=M_FLOOR):
    """Run the whole chain (Newmark) and return peak roof displacement (m), peak drift (m), peak base shear (N), first period (s)."""
    t, ag = synthetic_ground(seed=seed, t_dom=t_dom, pga=pga)
    dt = t[1] - t[0]
    M, K = shear_building(n, m, k)
    w, phi = modes(M, K)
    C = damping_matrix(M, phi, w, zeta)
    u, _, _ = newmark(M, C, K, ag, dt)
    k1 = np.broadcast_to(np.asarray(k, float), (n,))[0]
    return {"roof": float(np.abs(u[:, -1]).max()), "drift": float(np.abs(storey_drift(u)).max()),
            "shear": float(np.abs(base_shear(u, k1)).max()), "T1": float(2 * np.pi / w[0])}


def with_damper(M, K, w1, mass_ratio=0.02, zeta_d=0.10):
    """Stretch idea: a tuned mass damper on the roof. Adds one more mass (mass_ratio * total mass) joined to the roof
    by a spring tuned to the first mode and a dashpot. Returns (M2, C2, K2) for the building plus damper."""
    n = M.shape[0]
    md = mass_ratio * np.trace(M)
    kd = md * w1 ** 2
    cd = 2 * zeta_d * md * w1
    w, phi = modes(M, K)
    C = damping_matrix(M, phi, w)
    M2 = np.zeros((n + 1, n + 1)); K2 = np.zeros((n + 1, n + 1)); C2 = np.zeros((n + 1, n + 1))
    M2[:n, :n] = M; K2[:n, :n] = K; C2[:n, :n] = C
    M2[n, n] = md
    for A, c in ((K2, kd), (C2, cd)):
        A[n - 1, n - 1] += c; A[n, n] += c; A[n - 1, n] -= c; A[n, n - 1] -= c
    return M2, C2, K2


def resonant_ground(w, dur=24.0, dt=0.01, a0=2.0):
    """A pure sine at angular frequency w (rad/s) with a 3 s fade-in: the worst case for a building with that frequency."""
    t = np.arange(0.0, dur + 0.5 * dt, dt)
    return t, a0 * np.sin(w * t) * np.clip(t / 3.0, 0, 1)


def soft_storey_stiffness(n=N_FLOORS, k=K_STOREY, which=0, factor=0.35):
    ks = np.full(n, k)
    ks[which] = factor * k
    return ks

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    M, K = shear_building()
    w, phi = modes(M, K)
    wa, sa = uniform_analytic(N_FLOORS, M_FLOOR, K_STOREY)
    print("periods (s):", np.round(periods(w)[:4], 4), " analytic:", np.round(periods(wa)[:4], 4))
    gamma, meff = participation(M, phi)
    print("effective mass fractions:", np.round(meff[:3] / meff.sum(), 3), " sum =", round(meff.sum() / np.trace(M), 6))

    t, ag = synthetic_ground()
    dt = t[1] - t[0]
    C = damping_matrix(M, phi, w)
    un, vn, an = newmark(M, C, K, ag, dt)
    um = modal_response(M, K, ag, dt)
    print("max |modal - Newmark| / peak:", np.abs(um - un).max() / np.abs(un).max())
    ein, estored = energy_balance(M, C, K, un, vn, ag, dt)
    print("energy in, accounted:", ein[-1], estored[-1])
    d = storey_drift(un)
    print("peak drift per storey (mm):", np.round(1e3 * np.abs(d).max(axis=0), 1))
    print("peak base shear (MN):", base_shear(un, K_STOREY).__abs__().max() / 1e6)

    fig, ax = plt.subplots(1, 3, figsize=(11, 3.8))
    y = np.arange(0, N_FLOORS + 1)
    for i in range(3):
        ax[0].plot(np.r_[0, phi[:, i] / phi[-1, i]], y, "o-", label=f"mode {i + 1}, T = {periods(w)[i]:.3f} s")
        ax[0].plot(np.r_[0, sa[:, i]], y, "k.", ms=3)
    ax[0].set_xlabel("mode shape (roof = 1)")
    ax[0].set_ylabel("floor number")
    ax[0].set_title("Mode shapes (dots: analytic)")
    ax[0].legend(fontsize=7)
    ax[0].grid(alpha=0.3)
    ax[1].plot(t, 1e3 * un[:, -1], label="Newmark, roof")
    ax[1].plot(t, 1e3 * um[:, -1], "--", label="modal, roof")
    ax[1].set_xlabel("time (s)")
    ax[1].set_ylabel("roof displacement (mm)")
    ax[1].set_title("Two solvers, one answer")
    ax[1].legend(fontsize=8)
    ax[1].grid(alpha=0.3)
    ax[2].plot(t, ag, lw=0.7)
    ax[2].set_xlabel("time (s)")
    ax[2].set_ylabel("ground acceleration (m/s^2)")
    ax[2].set_title("Synthetic record (not a real quake)")
    ax[2].grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(OUT / "modes_and_response.png", dpi=120)

    Ks = shear_building(k=soft_storey_stiffness())[1]
    ws, ps = modes(M, Ks)
    Cs = damping_matrix(M, ps, ws)
    us, _, _ = newmark(M, Cs, Ks, ag, dt)
    ds = np.abs(storey_drift(us)).max(axis=0)
    fig2, ax2 = plt.subplots(figsize=(5, 4.2))
    ax2.barh(np.arange(1, 11) - 0.2, 1e3 * np.abs(d).max(axis=0), 0.4, label="uniform")
    ax2.barh(np.arange(1, 11) + 0.2, 1e3 * ds, 0.4, label="soft storey 1 (35% stiffness)")
    ax2.set_xlabel("peak inter-storey drift (mm)")
    ax2.set_ylabel("storey")
    ax2.set_title("A weak storey collects the drift")
    ax2.legend(fontsize=8)
    ax2.grid(alpha=0.3)
    fig2.tight_layout()
    fig2.savefig(OUT / "soft_storey.png", dpi=120)
    print("soft storey 1 peak drift (mm):", 1e3 * ds[0], " uniform:", 1e3 * np.abs(d).max(axis=0)[0])

    periods_grid = np.linspace(0.1, 3.0, 59)
    sd = [sdof_peak(T, ZETA, ag, dt) for T in periods_grid]
    fig3, ax3 = plt.subplots(figsize=(5.5, 4))
    ax3.plot(periods_grid, 1e3 * np.array(sd))
    ax3.set_xlabel("oscillator period (s)")
    ax3.set_ylabel("peak displacement (mm)")
    ax3.set_title("Displacement response spectrum, 5% damping")
    ax3.grid(alpha=0.3)
    fig3.tight_layout()
    fig3.savefig(OUT / "response_spectrum.png", dpi=120)

## Where this stops
Real buildings yield, crack and twist, and real earthquakes are not one synthetic record. This
model is linear, planar and fixed at the base: use it to build intuition, never to design.

## Coding tasks
1. **Easy.** Run the lesson and print the first four periods of a 5-storey frame (use `shear_building(5)` and `modes`). Compare T1 with 0.1 N.
2. **Stretch.** Use `peaks(zeta=0.02)`, `peaks()` and `peaks(zeta=0.1)` to build a table of roof sway against damping, then explain why the effect is bigger from 2 to 5 percent than from 5 to 10 percent.
3. **Explain.** Stretch: use `with_damper` and `resonant_ground` to add a tuned mass damper of 5 percent of the mass on the roof, and explain in words why it reduces the sway of the first mode.

## Assessment
- **A. Conceptual.** Why does the first mode carry most of the mass in the response, and why does the second mode have a node (a floor that hardly moves)?
- **B. Numerical.** Two storeys with m = 100 000 kg and k = 1e8 N/m: find both frequencies and periods by hand, then say which mode is more dangerous for a ground period of 1 s.
- **C. Graph and simulation interpretation.** In the soft-storey bar chart, which storey carries the largest drift and how much more than in the uniform frame? What does the damage tell an engineer?
- **D. Design and optimisation challenge.** Design: the first storey is weaker than the others by a factor f (between 0.35 and 1). Find the smallest f in steps of 0.1 that keeps the first-storey drift at no more than 1.5 times the largest drift of the other storeys. Justify the trade-off.

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def _const_push_ok(L):
    """Constant ground acceleration a=1: u settles to -1/w^2; undamped it swings to -2/w^2."""
    w = 2 * np.pi
    a = np.ones(1001)
    q0, _ = L.sdof_response(w, 1e-9, a, 0.01)
    q1, _ = L.sdof_response(w, 0.1, a, 0.01)
    return abs(q0.min() + 2 / w ** 2) < 1e-4 and abs(q1[-1] + 1 / w ** 2) < 0.05 / w ** 2 and abs(q1.min()) > abs(q1[-1])


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    # ---- worked example: a two-storey building by hand ------------------------------------------
    m, k = L.M_FLOOR, L.K_STOREY
    M2, K2 = L.shear_building(2, m, k)
    w2, p2 = L.modes(M2, K2)
    lam = np.array([(3 - np.sqrt(5)) / 2, (3 + np.sqrt(5)) / 2]) * k / m
    checks.append((f"worked example: 2 storeys give w^2 = (3 -/+ sqrt 5)/2 k/m, w = 25.854 and 67.687 rad/s, T = 0.2430 and 0.0928 s (got {w2[0]:.3f}, {w2[1]:.3f})",
                   np.allclose(w2 ** 2, lam) and abs(w2[0] - 25.854) < 5e-3 and abs(w2[1] - 67.687) < 5e-3
                   and abs(2 * np.pi / w2[0] - 0.2430) < 5e-5 and abs(2 * np.pi / w2[1] - 0.0928) < 5e-5))
    s1, s2 = p2[:, 0] / p2[0, 0], p2[:, 1] / p2[0, 1]
    checks.append((f"worked example: mode shapes (1, 1.618) and (1, -0.618) (got {s1[1]:.3f}, {s2[1]:.3f})",
                   abs(s1[1] - (1 + np.sqrt(5)) / 2) < 1e-9 and abs(s1[1] - 1.618) < 5e-4 and abs(s2[1] + 0.618) < 5e-4))
    _, me2 = L.participation(M2, p2)
    checks.append((f"worked example: effective mass fractions 94.7% and 5.3% (got {me2[0] / me2.sum():.3f}, {me2[1] / me2.sum():.3f})",
                   abs(me2[0] / me2.sum() - 0.9472) < 5e-4 and abs(me2[1] / me2.sum() - 0.0528) < 5e-4))

    # ---- matrices -------------------------------------------------------------------------------
    M, K = L.shear_building()
    checks.append(("K is symmetric and tridiagonal; K[0,0] = 2k, last diagonal = k",
                   np.allclose(K, K.T) and np.allclose(np.triu(K, 2), 0) and K[0, 0] == 2 * k and K[-1, -1] == k and K[0, 1] == -k))
    checks.append(("row sums of K are zero except the ground-connected first row (k)", np.allclose(K.sum(axis=1), [k] + [0.0] * 9)))
    w, phi = L.modes(M, K)
    checks.append(("mode shapes are mass-orthonormal: phi^T M phi = I", np.allclose(phi.T @ M @ phi, np.eye(10), atol=1e-9)))
    checks.append(("K phi = w^2 M phi holds for every mode", np.allclose(K @ phi, M @ phi @ np.diag(w ** 2), rtol=1e-8, atol=1e-3)))

    # ---- validation against the analytic uniform-building result --------------------------------
    wa, sa = L.uniform_analytic(10, m, k)
    T = 2 * np.pi / w
    checks.append((f"periods of the 10-storey building: T1 = 1.0049 s, T2 = 0.3375 s, T3 = 0.2056 s (got {T[0]:.4f}, {T[1]:.4f}, {T[2]:.4f})",
                   abs(T[0] - 1.0049) < 5e-5 and abs(T[1] - 0.3375) < 5e-5 and abs(T[2] - 0.2056) < 5e-5))
    checks.append(("numeric frequencies match 2 sqrt(k/m) sin((2i-1) pi / (2(2N+1))) to 1e-12 (relative)", np.allclose(w, wa, rtol=1e-12)))
    shapes = phi / phi[-1, :]
    checks.append(("numeric mode shapes match sin((2i-1) j pi / (2N+1)) to 1e-9", np.allclose(shapes, sa, atol=1e-9)))
    checks.append(("first mode has no sign change, mode 2 has one, mode 3 has two (nodes)",
                   [int((np.diff(np.sign(phi[np.abs(phi[:, i]) > 1e-9 * np.abs(phi[:, i]).max(), i])) != 0).sum()) for i in range(3)] == [0, 1, 2]))
    checks.append(("rule of thumb T = 0.1 N = 1.0 s is close to the computed T1 = 1.005 s", abs(0.1 * 10 - T[0]) < 0.01))

    # ---- participation and effective mass -------------------------------------------------------
    gamma, meff = L.participation(M, phi)
    frac = meff / np.trace(M)
    checks.append((f"effective masses add up to the total mass 2.0e6 kg (sum = {meff.sum():.6g})", abs(meff.sum() - np.trace(M)) < 1e-6 * np.trace(M)))
    checks.append((f"first mode carries 84.8%, second 9.1%, third 3.1% of the mass (got {frac[0]:.3f}, {frac[1]:.3f}, {frac[2]:.3f})",
                   abs(frac[0] - 0.848) < 5e-4 and abs(frac[1] - 0.0914) < 5e-4 and abs(frac[2] - 0.0309) < 5e-4))
    checks.append(("first three modes together carry more than 97 percent", frac[:3].sum() > 0.97))

    # ---- ground record --------------------------------------------------------------------------
    t, ag = L.synthetic_ground()
    dt = t[1] - t[0]
    ag_b = L.synthetic_ground()[1]
    checks.append(("synthetic record is repeatable (same seed gives the same numbers) and differs for another seed",
                   np.array_equal(ag, ag_b) and not np.allclose(ag, L.synthetic_ground(seed=8)[1])))
    checks.append((f"record: 24 s long, dt = 0.01 s, peak exactly 2.0 m/s^2 (got {np.abs(ag).max():.6f})",
                   abs(t[-1] - 24.0) < 1e-9 and abs(dt - 0.01) < 1e-12 and abs(np.abs(ag).max() - 2.0) < 1e-9))
    spec = np.abs(np.fft.rfft(ag))
    freq = np.fft.rfftfreq(ag.size, dt)
    fdom = freq[np.argmax(spec[1:]) + 1]
    checks.append((f"record energy peaks near 1 Hz (dominant period about 1 s): spectral peak at {fdom:.2f} Hz", 0.6 < fdom < 1.5))

    # ---- the two solvers ------------------------------------------------------------------------
    C = L.damping_matrix(M, phi, w)
    gen = phi.T @ C @ phi
    checks.append(("damping matrix gives zeta = 0.05 in every mode", np.allclose(np.diag(gen) / (2 * w), 0.05) and np.allclose(gen, np.diag(np.diag(gen)), atol=1e-6)))
    un, vn, an = L.newmark(M, C, K, ag, dt)
    um = L.modal_response(M, K, ag, dt)
    rel = np.abs(um - un).max() / np.abs(un).max()
    checks.append((f"modal superposition and Newmark agree to within 1 percent of the peak (got {100 * rel:.2f} percent)", rel < 0.01))
    checks.append(("exact one-mode step is right in a simple case: a constant push settles to -1/w^2 and (undamped) swings to -2/w^2", _const_push_ok(L)))
    t2, ag2 = L.synthetic_ground(dt=0.005)
    u2, _, _ = L.newmark(M, C, K, ag2, 0.005)
    diff = np.abs(u2[::2][:un.shape[0]] - un).max() / np.abs(un).max()
    checks.append((f"Newmark with half the time step changes the answer by only {100 * diff:.2f} percent", diff < 0.005))
    checks.append(("Newmark parameters: average acceleration beta = 1/4 and gamma = 1/2 are the defaults", L.newmark.__defaults__ == (0.25, 0.5)))
    C0 = 0 * C
    u0, v0, _ = L.newmark(M, C0, K, ag, dt)
    ein0, es0 = L.energy_balance(M, C0, K, u0, v0, ag, dt)
    checks.append((f"undamped: input energy equals stored energy to 0.5 percent ({ein0[-1]:.4g} J vs {es0[-1]:.4g} J)",
                   abs(ein0[-1] - es0[-1]) / abs(ein0[-1]) < 5e-3))
    ein, eacc = L.energy_balance(M, C, K, un, vn, ag, dt)
    checks.append((f"damped: input 3.641e6 J equals kinetic + strain + dissipated to 1e-4 (got {ein[-1]:.4g}, {eacc[-1]:.4g})",
                   abs(ein[-1] - eacc[-1]) / ein[-1] < 1e-4 and abs(ein[-1] - 3.641e6) < 5e3))

    # ---- responses ------------------------------------------------------------------------------
    roof = np.abs(un[:, -1]).max()
    drift = np.abs(L.storey_drift(un)).max(axis=0)
    shear = np.abs(L.base_shear(un, k)).max()
    checks.append((f"peak roof displacement 0.172 m (got {roof:.4f}); peak drift 25.1 mm in storey 1, 3.9 mm in storey 10 (got {1e3 * drift[0]:.1f}, {1e3 * drift[-1]:.1f})",
                   abs(roof - 0.1719) < 5e-4 and int(np.argmax(drift)) == 0 and abs(1e3 * drift[0] - 25.1) < 0.05 and abs(1e3 * drift[-1] - 3.9) < 0.05))
    checks.append((f"drift ratio 25.1 mm / 3.5 m = 0.72 percent; peak base shear 8.77 MN (got {shear / 1e6:.2f}), 0.45 of the weight 19.6 MN",
                   abs(drift[0] / L.STOREY_HEIGHT * 100 - 0.72) < 0.01 and abs(shear / 1e6 - 8.77) < 0.01
                   and abs(shear / (2e6 * 9.81) - 0.447) < 0.002))
    sm = np.abs(L.base_shear(um, k)).max()
    checks.append(("base shear from modal superposition is within 0.3 percent of Newmark", abs(sm - shear) / shear < 0.003))
    roof1 = np.abs(L.modal_response(M, K, ag, dt, nmodes=1)[:, -1]).max()
    checks.append((f"mode 1 alone gives the peak roof displacement within 1 percent (got {roof1:.4f} m)", abs(roof1 - roof) / roof < 0.01))
    sd1 = L.sdof_peak(1.0, 0.05, ag, dt)
    sd_short = L.sdof_peak(0.2, 0.05, ag, dt)
    checks.append((f"response spectrum: oscillator with T = 1.0 s moves {1e3 * sd1:.0f} mm, T = 0.2 s only {1e3 * sd_short:.0f} mm", sd1 > 10 * sd_short))

    # ---- sensitivity ----------------------------------------------------------------------------
    p_base = L.peaks()
    checks.append(("peaks() reproduces the Newmark run (roof, drift, T1 = 1.0049 s)",
                   abs(p_base["roof"] - roof) < 1e-12 and abs(p_base["T1"] - 1.0049) < 5e-5))
    z2, z10 = L.peaks(zeta=0.02), L.peaks(zeta=0.10)
    checks.append((f"damping sensitivity: roof peak {1e3 * z2['roof']:.0f} mm at 2%, {1e3 * p_base['roof']:.0f} mm at 5%, {1e3 * z10['roof']:.0f} mm at 10%",
                   z2["roof"] > p_base["roof"] > z10["roof"] and abs(1e3 * z2["roof"] - 259) < 1 and abs(1e3 * z10["roof"] - 163) < 1))
    s08, s12 = L.peaks(k=0.8 * k), L.peaks(k=1.2 * k)
    checks.append((f"stiffness sensitivity: 20 percent softer gives T1 = {s08['T1']:.3f} s and roof {1e3 * s08['roof']:.0f} mm; 20 percent stiffer gives T1 = {s12['T1']:.3f} s and roof {1e3 * s12['roof']:.0f} mm",
                   abs(s08["T1"] - 1.1235) < 5e-4 and abs(s12["T1"] - 0.9174) < 5e-4 and abs(1e3 * s08["roof"] - 274) < 1 and abs(1e3 * s12["roof"] - 191) < 1
                   and abs(s12["T1"] * np.sqrt(1.2) - p_base["T1"]) < 1e-9))

    # ---- failure case: soft storey --------------------------------------------------------------
    Ms, Ks = L.shear_building(k=L.soft_storey_stiffness())
    ws, ps = L.modes(Ms, Ks)
    Cs = L.damping_matrix(Ms, ps, ws)
    us, _, _ = L.newmark(Ms, Cs, Ks, ag, dt)
    ds = np.abs(L.storey_drift(us)).max(axis=0)
    checks.append((f"soft storey (storey 1 at 35 percent stiffness): storey-1 drift 95.3 mm, 3.8 times the uniform building (got {1e3 * ds[0]:.1f} mm, x{ds[0] / drift[0]:.2f})",
                   abs(1e3 * ds[0] - 95.3) < 0.1 and abs(ds[0] / drift[0] - 3.80) < 0.01 and int(np.argmax(ds)) == 0))
    shape1 = ps[:, 0] / ps[-1, 0]
    d1 = np.diff(np.r_[0, shape1])
    d0 = np.diff(np.r_[0, (lambda q: q[:, 0] / q[-1, 0])(phi)])
    checks.append((f"first-mode shape: storey 1 takes {100 * d1[0]:.1f} percent of the roof displacement when soft (against {100 * d0[0]:.1f} percent when uniform)",
                   abs(100 * d1[0] - 35.0) < 0.1 and abs(100 * d0[0] - 14.9) < 0.1))
    checks.append((f"the soft storey lengthens the first period (T1 1.0049 s -> {2 * np.pi / ws[0]:.3f} s)", abs(2 * np.pi / ws[0] - 1.176) < 0.001 and ws[0] < w[0]))

    # ---- stretch: tuned mass damper -------------------------------------------------------------
    tt, ah = L.resonant_ground(w[0])
    uh, _, _ = L.newmark(M, C, K, ah, 0.01)
    M3, C3, K3 = L.with_damper(M, K, w[0], 0.05, 0.15)
    u3, _, _ = L.newmark(M3, C3, K3, ah, 0.01)
    ratio = np.abs(u3[:, 9]).max() / np.abs(uh[:, 9]).max()
    checks.append((f"tuned mass damper (5 percent of the mass, tuned to mode 1) cuts the resonant roof sway to {100 * ratio:.0f} percent of the bare building",
                   ratio < 0.4 and abs(ratio - 0.31) < 0.02))
    checks.append(("damper adds one degree of freedom and keeps the matrices symmetric", M3.shape == (11, 11) and np.allclose(K3, K3.T) and np.allclose(C3, C3.T)))

    # ---- mini-project: stiffen the 10-storey frame ----------------------------------------------
    base = L.peaks()
    rows = {f: L.peaks(k=f * k) for f in (1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0)}
    ok_f = [f for f, r in rows.items() if r["roof"] <= 0.5 * base["roof"] and r["drift"] / L.STOREY_HEIGHT <= 0.005]
    checks.append((f"mini-project: smallest stiffness factor (steps of 0.5) with roof sway at most half of the original and drift ratio at most 0.5 percent is 3.0 (got {min(ok_f) if ok_f else None})",
                   bool(ok_f) and min(ok_f) == 3.0 and 1.0 not in ok_f))
    r3 = rows[3.0]
    checks.append((f"mini-project: factor 3 gives T1 = 0.580 s, roof {1e3 * r3['roof']:.0f} mm, drift ratio {100 * r3['drift'] / L.STOREY_HEIGHT:.2f} percent, base shear {r3['shear'] / 1e6:.2f} MN",
                   abs(r3["T1"] - 0.5802) < 5e-4 and abs(1e3 * r3["roof"] - 63.3) < 0.1 and abs(100 * r3["drift"] / L.STOREY_HEIGHT - 0.28) < 0.005
                   and abs(r3["shear"] / 1e6 - 10.37) < 0.01))
    checks.append(("mini-project trade-off: the stiffer building has the larger base shear (10.37 against 8.77 MN)", r3["shear"] > base["shear"]))
    checks.append(("mini-project: factor 1.5 does not help (roof 173 mm, no lower than the original 172 mm): a record is not one frequency",
                   abs(1e3 * rows[1.5]["roof"] - 173.3) < 0.1 and rows[1.5]["roof"] > base["roof"]))
    cuts = [L.peaks(k=3 * k, seed=s)["roof"] < 0.5 * L.peaks(seed=s)["roof"] for s in (1, 2, 3, 4, 5, 6)]
    checks.append(("mini-project: factor 3 more than halves the roof sway for each of six other random seeds (1 to 6)", all(cuts)))
    r = base["T1"] / 1.0
    checks.append(("first-mode amplification D(T1 / Tg) for the original frame is 9.9 (near resonance), and 1.50 at factor 3",
                   abs(1 / np.sqrt((1 - r ** 2) ** 2 + (0.1 * r) ** 2) - 9.9) < 0.05
                   and abs(1 / np.sqrt((1 - r3["T1"] ** 2) ** 2 + (0.1 * r3["T1"]) ** 2) - 1.50) < 0.01))
    Mb, Kb = L.shear_building(2, 1e5, 1e8)
    wb, _ = L.modes(Mb, Kb)
    checks.append(("B: two storeys with m = 100000 kg, k = 1e8 N/m: w = 19.544 and 51.167 rad/s, T = 0.3215 and 0.1228 s",
                   abs(wb[0] - 19.544) < 5e-4 and abs(wb[1] - 51.167) < 5e-4 and abs(2 * np.pi / wb[0] - 0.3215) < 5e-5 and abs(2 * np.pi / wb[1] - 0.1228) < 5e-5))
    res = {}
    for f in (0.35, 0.6, 0.7, 1.0):
        Mf, Kf = L.shear_building(k=L.soft_storey_stiffness(factor=f))
        wf, pf = L.modes(Mf, Kf)
        uf, _, _ = L.newmark(Mf, L.damping_matrix(Mf, pf, wf), Kf, ag, dt)
        df = np.abs(L.storey_drift(uf)).max(axis=0)
        res[f] = (df[0], df[0] / df[1:].max())
    checks.append((f"D: weak storey at 0.7 gives storey-1 drift 43.5 mm, 1.46 times the largest other storey (within 1.5); at 0.6 it is 1.71 times",
                   abs(1e3 * res[0.7][0] - 43.5) < 0.1 and abs(res[0.7][1] - 1.46) < 0.01 and abs(res[0.6][1] - 1.71) < 0.01
                   and abs(1e3 * res[0.6][0] - 55.4) < 0.1 and abs(res[0.35][1] - 2.97) < 0.01 and abs(res[1.0][1] - 1.01) < 0.01))
    checks.append((f"frequency ratios: T1 / T2 = 2.98 for ten storeys and 2.62 for two storeys; storey-1 drift ratio {1e3 * ds[0] / 3.5:.0f} -> 2.7 percent with a soft storey",
                   abs(w[1] / w[0] - 2.977) < 5e-3 and abs(w2[1] / w2[0] - 2.618) < 5e-3 and abs(ds[0] / L.STOREY_HEIGHT * 100 - 2.72) < 0.01))
    M1, K1 = L.shear_building(1, m, k)
    w1, _ = L.modes(M1, K1)
    checks.append(("edge: one storey reduces to w = sqrt(k/m) from Session 2", abs(w1[0] - np.sqrt(k / m)) < 1e-9))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
